# Day 2: RLHF, Model Alignment, DPO, and GRPO

The notebook has two parts. **Part A** explains the theory through three distinct alignment approaches. **Part B** contains brief TensorFlow demonstrations of their real training signals.

# Course overview

```text
Pretrained model → SFT
                     ├── Classical RLHF → Reward model → PPO
                     ├── DPO → Chosen/rejected response pairs
                     └── GRPO → Groups of generated responses and rewards
```

Each method is explained immediately before its corresponding TensorFlow implementation. GRPO is theory-only in this notebook.

In [21]:
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)

print('All examples use small TensorFlow models for classroom clarity.')

TensorFlow: 2.20.0
All examples use small TensorFlow models for classroom clarity.


# Section 1 — Classical RLHF

## Start with an example

**Prompt:** `Explain photosynthesis to a 10-year-old.`

- Response A is short, clear, and correct.
- Response B is difficult and contains an error.

A human chooses A. A reward model learns from comparisons like this. Later, the policy generates many new responses, the reward model scores them, and an RL optimizer such as PPO improves the policy.

```text
Human comparisons → Train reward model → Score new responses → PPO updates policy
```

## 1.1 Why alignment and SFT are needed

A pretrained model learns next-token prediction, but not necessarily what people consider helpful, honest, and harmless. SFT trains the model on high-quality prompt-response demonstrations and creates an instruction-following starting policy.

SFT cannot cover every future prompt or rank every plausible answer. RLHF adds preference information after SFT.

## 1.2 Classical RLHF flow

```text
1. Pretrained model
        ↓
2. Supervised Fine-Tuning (SFT)
        ↓
3. Generate candidate responses
        ↓
4. Humans compare the candidates
        ↓
5. Train a reward model from the comparisons
        ↓
6. Policy generates new responses
        ↓
7. Reward model scores them
        ↓
8. PPO updates the policy; KL limits drift
        ↓
9. Repeat
```

Human preferences train the reward model because humans cannot continuously score the enormous number of responses required during RL training. The reward model is a scalable, imperfect approximation of human judgement.

## 1.3 Policy and preference data

A **policy model** is the language model being updated. Its **policy** is its strategy for choosing the next token—represented by probabilities over possible tokens.

After `The capital of France is`, a policy might assign `Paris: 90%`, `London: 4%`, `Berlin: 2%`, and other tokens: `4%`.

The notation $\pi(a_t \mid s_t)$ means the probability of choosing token $a_t$ when $s_t$ contains the prompt and tokens generated so far.

A preference record contains:

- `prompt`: the user request
- `chosen`: the human-preferred response
- `rejected`: the less-preferred response

Chosen means preferred; it is not an automatic guarantee of perfect correctness.

## 1.4 Reward-model training

The reward model maps `prompt + response` to one score. It is trained so the chosen response scores higher than the rejected response:

$$L_{RM}=-\log\sigma(r_c-r_r)$$

- $L_{RM}$: loss minimized during reward-model training
- $r_c$: chosen-response score
- $r_r$: rejected-response score
- $\sigma$: sigmoid function
- $\log$: natural logarithm

If $r_c$ is much larger than $r_r$, the loss is small. After training, the reward model is normally frozen and used as an automatic scorer.

## 1.5 Practical — Prepare data for Reward Model + PPO

## Step 1: Prepare human-preference data

- `prompt`: question given to the model
- `chosen`: response preferred by the evaluator
- `rejected`: less-preferred response

Chosen means preferred; it is not an automatic guarantee of perfect correctness.

In [22]:
preference_data = [
    {
        'prompt': 'What is AI?',
        'chosen': 'AI enables machines to perform tasks that normally require human intelligence.',
        'rejected': 'AI is something computers do.',
    },
    {
        'prompt': 'What is Python?',
        'chosen': 'Python is a programming language used in web development, data science, and AI.',
        'rejected': 'Python is only a snake.',
    },
    {
        'prompt': 'Explain gravity simply.',
        'chosen': 'Gravity is a force that pulls objects with mass toward each other.',
        'rejected': 'Gravity is when things fall.',
    },
    {
        'prompt': 'Why is the sky blue?',
        'chosen': 'Air scatters blue light from the Sun more strongly than most other colours.',
        'rejected': 'The sky is painted blue.',
    },
]

chosen_text = tf.constant([
    row['prompt'] + ' ' + row['chosen'] for row in preference_data
])
rejected_text = tf.constant([
    row['prompt'] + ' ' + row['rejected'] for row in preference_data
])
print('Number of preference pairs:', len(preference_data))

Number of preference pairs: 4


## 1.6 Practical — Train the reward model

The model gives one score to `prompt + response`. Pairwise loss teaches it to score chosen responses higher than rejected responses.

In [23]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=1000, output_mode='int', output_sequence_length=30
)
vectorizer.adapt(tf.concat([chosen_text, rejected_text], axis=0))

reward_model = tf.keras.Sequential([
    tf.keras.Input(shape=(), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(1000, 16, mask_zero=True),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(1),
])
reward_optimizer = tf.keras.optimizers.Adam(0.01)

for _ in range(100):
    with tf.GradientTape() as tape:
        chosen_score = reward_model(chosen_text, training=True)
        rejected_score = reward_model(rejected_text, training=True)
        reward_loss = -tf.reduce_mean(
            tf.math.log_sigmoid(chosen_score - rejected_score)
        )
    gradients = tape.gradient(reward_loss, reward_model.trainable_variables)
    reward_optimizer.apply_gradients(zip(gradients, reward_model.trainable_variables))

print('Reward-model loss:', round(float(reward_loss.numpy()), 4))
for i in range(len(preference_data)):
    print(f'Pair {i+1}: chosen={chosen_score[i,0]:+.2f}, rejected={rejected_score[i,0]:+.2f}')

Reward-model loss: 0.0026
Pair 1: chosen=+2.56, rejected=-3.91
Pair 2: chosen=+1.46, rejected=-4.20
Pair 3: chosen=+2.06, rejected=-3.93
Pair 4: chosen=+1.75, rejected=-4.15


## 1.7 Practical — Calculate rewards and advantages

For each prompt, action 0 selects the chosen candidate and action 1 selects the rejected candidate. The mean score for that prompt acts as a simple baseline.

$$A=R-\text{baseline}$$

A positive advantage means better than expected; a negative advantage means worse than expected.

In [24]:
reward_model.trainable = False
chosen_rewards = tf.squeeze(reward_model(chosen_text), axis=1)
rejected_rewards = tf.squeeze(reward_model(rejected_text), axis=1)
rewards = tf.stack([chosen_rewards, rejected_rewards], axis=1)
advantages = rewards - tf.reduce_mean(rewards, axis=1, keepdims=True)

print('Rewards [chosen, rejected]:\n', np.round(rewards.numpy(), 2))
print('Advantages:\n', np.round(advantages.numpy(), 2))

Rewards [chosen, rejected]:
 [[ 2.57 -3.92]
 [ 1.46 -4.21]
 [ 2.06 -3.94]
 [ 1.75 -4.16]]
Advantages:
 [[ 3.24 -3.24]
 [ 2.84 -2.84]
 [ 3.   -3.  ]
 [ 2.95 -2.95]]


## 1.8 PPO theory matching the implementation

The classroom policy does not generate tokens. For each prompt $i$, it chooses between two candidate responses:

- action $j=0$: choose the preferred candidate;
- action $j=1$: choose the rejected candidate.

### 1. Policy logits and probabilities

`policy_logits` contains two trainable scores for every prompt. `softmax` converts the scores into probabilities:

$$p_{i,j}=\operatorname{softmax}(z_i)_j$$

- $z_i$: the two trainable logits for prompt $i$
- $p_{i,j}$: probability of selecting candidate $j$ for prompt $i$

The code initializes both logits to zero, so each candidate begins with probability $0.5$.

### 2. Freeze the old probabilities

Before PPO updates begin, the code saves:

$$p^{\mathrm{old}}_{i,j}=\operatorname{stop\_gradient}(p_{i,j})$$

`stop_gradient` makes the old probabilities a fixed comparison point. PPO updates only the new policy logits.

### 3. Calculate advantages

The frozen reward model produces $R_{i,j}$ for each candidate. The code uses the mean reward of the two candidates as a simple baseline:

$$A_{i,j}=R_{i,j}-\frac{R_{i,0}+R_{i,1}}{2}$$

Positive $A_{i,j}$ means better than the prompt's average candidate; negative means worse.

### 4. Calculate the PPO probability ratio

During training, `softmax(policy_logits)` produces new probabilities. The ratio is:

$$r_{i,j}=\frac{p^{\mathrm{new}}_{i,j}}{p^{\mathrm{old}}_{i,j}}$$

A ratio above 1 means the candidate became more likely; below 1 means it became less likely.

### 5. Clip the ratio

With clipping parameter $\epsilon=0.2$:

$$\widehat r_{i,j}=\operatorname{clip}(r_{i,j},1-\epsilon,1+\epsilon)$$

Therefore the clipped ratio stays between $0.8$ and $1.2$.

### 6. PPO objective and loss

The code calculates exactly:

$$O_{i,j}=\min\left(r_{i,j}A_{i,j},\widehat r_{i,j}A_{i,j}\right)$$

$$L_{\mathrm{PPO}}=-\operatorname{mean}_{i,j}(O_{i,j})$$

TensorFlow minimizes `ppo_loss`. The minus sign converts the objective that PPO wants to maximize into a loss that the optimizer minimizes. Gradients update only `policy_logits`.

This is a small candidate-selection version of PPO. In an LLM, the same ratio and clipping are calculated for generated-token probabilities.

## 1.9 Practical — Apply the matching PPO loss

The policy begins with a 50–50 probability for both candidates. PPO uses the frozen old probabilities, the advantages, and clipping. We expect the chosen-response probabilities to increase.

In [25]:
policy_logits = tf.Variable(tf.zeros((len(preference_data), 2)))
old_probabilities = tf.stop_gradient(tf.nn.softmax(policy_logits, axis=1))
ppo_optimizer = tf.keras.optimizers.SGD(0.2)
epsilon = 0.2

before = old_probabilities[:, 0].numpy()
for _ in range(20):
    with tf.GradientTape() as tape:
        new_probabilities = tf.nn.softmax(policy_logits, axis=1)
        ratio = new_probabilities / old_probabilities
        clipped_ratio = tf.clip_by_value(ratio, 1-epsilon, 1+epsilon)
        ppo_objective = tf.minimum(ratio*advantages, clipped_ratio*advantages)
        ppo_loss = -tf.reduce_mean(ppo_objective)
    gradient = tape.gradient(ppo_loss, policy_logits)
    ppo_optimizer.apply_gradients([(gradient, policy_logits)])

after = tf.nn.softmax(policy_logits, axis=1)[:, 0].numpy()
print('Chosen probability before PPO:', np.round(before, 3))
print('Chosen probability after PPO: ', np.round(after, 3))

Chosen probability before PPO: [0.5 0.5 0.5 0.5]
Chosen probability after PPO:  [0.618 0.604 0.61  0.608]


## 1.10 Reference model and KL penalty in full RLHF

**Scope note:** the preceding classroom PPO code implements exactly the clipped PPO equations in Section 1.8. It does not add a reference-model KL term. The KL penalty below is an additional component commonly used when PPO updates a full language model.


Optimizing only reward can cause reward hacking or destroy useful SFT behaviour. A frozen reference model represents the starting SFT policy.

$$J = R - \beta D_{\mathrm{KL}}\!\left(\pi_{\mathrm{policy}} \;\Vert\; \pi_{\mathrm{reference}}\right)$$

- $J$: objective to maximize
- $R$: reward-model score
- $D_{\mathrm{KL}}$: divergence measuring policy change
- $\pi_{\mathrm{policy}}$: current trainable policy
- $\pi_{\mathrm{reference}}$: frozen SFT policy
- $\beta$: strength of the KL penalty

Larger $\beta$ keeps the policy closer to the reference; smaller $\beta$ allows more change.

## 1.11 RLHF and PPO challenges

- Human comparisons are expensive and can be inconsistent.
- Reward models can learn bias and incorrect shortcuts.
- PPO is sensitive to learning rate, clipping, reward scale, and KL strength.
- Multiple models require considerable memory and computation.
- A high predicted reward does not guarantee real user preference.

# Section 2 — Direct Preference Optimization (DPO)

## Start with an example

**Prompt:** `What is gradient descent?`

- **Chosen:** `It repeatedly adjusts parameters in a direction that reduces prediction error.`
- **Rejected:** `It sorts rows in a database.`

DPO uses this pair directly to increase the policy's relative preference for the chosen response. It does not train a separate reward model or run PPO.

```text
Chosen/rejected pair → DPO loss → Update policy
```

## 2.1 DPO theory matching the implementation

For each prompt $i$, the policy scores a chosen response and a rejected response. `log_softmax` converts those two scores into pair log probabilities:

$$\ell^{\mathrm{policy}}_{i,c}=\log p^{\mathrm{policy}}_{i,c}, \qquad \ell^{\mathrm{policy}}_{i,r}=\log p^{\mathrm{policy}}_{i,r}$$

The frozen reference model calculates the same quantities. The two preference margins are:

$$m^{\mathrm{policy}}_i=\ell^{\mathrm{policy}}_{i,c}-\ell^{\mathrm{policy}}_{i,r}$$

$$m^{\mathrm{reference}}_i=\ell^{\mathrm{reference}}_{i,c}-\ell^{\mathrm{reference}}_{i,r}$$

The implementation uses exactly this loss:

$$L_{\mathrm{DPO}}=-\operatorname{mean}_i\left[\log\sigma\left(\beta(m^{\mathrm{policy}}_i-m^{\mathrm{reference}}_i)\right)\right]$$

- $c$: chosen response
- $r$: rejected response
- $\ell$: log probability
- $m$: chosen-minus-rejected preference margin
- $\sigma$: sigmoid
- $\beta$: strength of the preference comparison

Minimizing the loss increases the chosen margin of the trainable policy relative to the frozen reference model.

## 2.2 DPO workflow and limitations

1. Start from an SFT policy and frozen reference copy.
2. Load prompt/chosen/rejected examples.
3. Calculate policy and reference sequence log probabilities.
4. Calculate DPO loss.
5. Update only the policy, possibly through LoRA adapters.
6. Evaluate quality, safety, and capability regressions.

DPO is simpler than PPO-based RLHF but depends strongly on preference-pair quality and cannot automatically explore and score new responses during offline training.

## 2.3 Practical — Create policy and frozen reference

DPO uses the same text preference pairs but does not use the reward model or PPO. We create:

- a trainable policy that scores each text response; and
- a frozen reference model copied from the policy before training.

In [26]:
def make_text_policy():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(), dtype=tf.string),
        vectorizer,
        tf.keras.layers.Embedding(1000, 16, mask_zero=True),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(1),
    ])

policy = make_text_policy()
reference = make_text_policy()
policy(chosen_text); reference(chosen_text)  # build both models
reference.set_weights(policy.get_weights())
reference.trainable = False
print('Policy parameters:', policy.count_params())

Policy parameters: 16017


## 2.4 Practical — Apply the matching DPO loss

For this small two-candidate demonstration, `log_softmax` converts chosen/rejected scores into pair log probabilities. DPO increases the policy preference margin relative to the frozen reference margin.

In [27]:
def pair_log_probabilities(model):
    chosen = tf.squeeze(model(chosen_text), axis=1)
    rejected = tf.squeeze(model(rejected_text), axis=1)
    return tf.nn.log_softmax(tf.stack([chosen, rejected], axis=1), axis=1)

dpo_optimizer = tf.keras.optimizers.Adam(0.01)
beta = 0.5
before = tf.exp(pair_log_probabilities(policy)[:, 0]).numpy()

reference_logp = tf.stop_gradient(pair_log_probabilities(reference))
for _ in range(100):
    with tf.GradientTape() as tape:
        policy_logp = pair_log_probabilities(policy)
        policy_margin = policy_logp[:, 0] - policy_logp[:, 1]
        reference_margin = reference_logp[:, 0] - reference_logp[:, 1]
        dpo_loss = -tf.reduce_mean(tf.math.log_sigmoid(
            beta * (policy_margin - reference_margin)
        ))
    gradients = tape.gradient(dpo_loss, policy.trainable_variables)
    dpo_optimizer.apply_gradients(zip(gradients, policy.trainable_variables))

after = tf.exp(pair_log_probabilities(policy)[:, 0]).numpy()
print('Chosen probability before DPO:', np.round(before, 3))
print('Chosen probability after DPO: ', np.round(after, 3))
print('Final DPO loss:', round(float(dpo_loss.numpy()), 4))

Chosen probability before DPO: [0.502 0.504 0.504 0.502]
Chosen probability after DPO:  [1. 1. 1. 1.]
Final DPO loss: 0.0065


# Section 3 — Group Relative Policy Optimization (GRPO)

## Start with an example

**Prompt:** `What is 6 × 7? Use the format Answer: number.`

| Response | Reward |
|---|---:|
| `Answer: 42` | 1.2 |
| `42` | 1.0 |
| `Answer: 41` | 0.2 |
| `I do not know` | 0.0 |

GRPO compares each response with the other responses in its group.

## 3.1 Rewards and group-relative advantages

For one prompt, generate $G$ responses $y_1,\ldots,y_G$ and calculate rewards $R_1,\ldots,R_G$.

$$A_i = \frac{R_i - \mu_R}{\sigma_R + \epsilon}$$

- $A_i$: relative advantage of response $i$
- $R_i$: its reward
- $\mu_R$: mean group reward
- $\sigma_R$: standard deviation of group rewards
- $\epsilon$: small value preventing division by zero

$A_i>0$ means above average, $A_i<0$ means below average, and $A_i \approx 0$ means near average. No separate value model is needed because the group provides the baseline.

## 3.2 How $A_i$ trains the policy

A simplified policy-gradient loss is:

$$L_{\mathrm{policy}} = -\frac{1}{G} \sum_{i=1}^{G} A_i \log \pi_{\theta}(y_i \mid x)$$

- Positive $A_i$: increase the response probability.
- Negative $A_i$: decrease the response probability.
- Near-zero $A_i$: make little change.

For generated token $t$ in response $i$, GRPO can compare current and old probabilities:

$$q_{i,t} = \frac{\pi_{\theta}(a_{i,t} \mid s_{i,t})}{\pi_{\mathrm{old}}(a_{i,t} \mid s_{i,t})}$$

A PPO-style clipped objective limits large changes. A reference-model KL penalty may also be used.

## 3.3 GRPO workflow and reward design

1. Generate a group of responses for each prompt.
2. Score every response with reward functions, verifiers, or a reward model.
3. Normalize group rewards to obtain $A_i$.
4. Weight response token probabilities using $A_i$.
5. Apply clipping and optional KL control.
6. Calculate gradients and update the policy.
7. Repeat with newly generated groups.

Rewards may measure correctness, code-test success, format compliance, or model-based judgement. Poor reward design leads to reward hacking.

# Final comparison and evaluation

| Method | Signal | Online generation? | Separate reward model? | Complexity |
|---|---|---|---|---|
| PPO-based RLHF | Learned reward | Yes | Yes | High |
| DPO | Chosen/rejected pairs | No | No | Lower |
| GRPO | Group-relative rewards | Yes | Not necessarily | Medium to high |

Evaluation should combine task performance, human preference, instruction following, safety, factuality, hallucination tests, response diversity, and capability-regression tests. No single metric proves alignment.

# Notebook recap

### PPO project

```text
Preference pairs → Reward model → Rewards → Advantages → Clipped PPO update
```

### DPO project

```text
Preference pairs → Policy and frozen reference → DPO loss → Policy update
```

These examples train real TensorFlow parameters through the stated losses. A full generative LLM applies the objectives to token-level sequence log probabilities and requires substantially more compute.